In [16]:
!pip install -q transformers datasets sentence-transformers accelerate
!pip install -q nltk
!pip install -q bitsandbytes

In [17]:
import os
import re
import torch
import numpy as np
import nltk
from datasets import load_dataset
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Trainer, TrainingArguments, DataCollatorForSeq2Seq,
    pipeline, BertForNextSentencePrediction
)

nltk.download('punkt', quiet=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [18]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = "/content/drive/MyDrive/arabic_summarization"
!mkdir -p {SAVE_DIR}/extractive
!mkdir -p {SAVE_DIR}/arata5_results

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
ds = load_dataset("arbml/AraSum")
print("Dataset loaded:", ds)
print("Train size:", len(ds["train"]))

Dataset loaded: DatasetDict({
    train: Dataset({
        features: ['index', 'summary', 'article'],
        num_rows: 49603
    })
})
Train size: 49603


In [20]:
model_name = "UBC-NLP/AraT5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)
print("Tokenizer and model loaded")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Tokenizer and model loaded


In [21]:
def preprocess(example):
    input_text = "summarize: " + example["article"]
    target_text = example["summary"]

    model_inputs = tokenizer(input_text, max_length=512, truncation=True)
    labels = tokenizer(target_text, max_length=128, truncation=True)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

In [22]:
tokenized_ds = ds.map(preprocess, remove_columns=ds["train"].column_names)
print("Tokenized dataset size:", len(tokenized_ds["train"]))

Tokenized dataset size: 49603


In [23]:
print("First sample's labels (after -100 masking):\n", tokenized_ds["train"][0]["labels"])
print("Second sample's labels (after -100 masking):\n", tokenized_ds["train"][1]["labels"])

print("\nDecoded summary for first sample's labels:\n", tokenizer.decode([t for t in tokenized_ds["train"][0]["labels"] if t != -100], skip_special_tokens=True))
print("Decoded summary for second sample's labels:\n", tokenizer.decode([t for t in tokenized_ds["train"][1]["labels"] if t != -100], skip_special_tokens=True))

First sample's labels (after -100 masking):
 [1399, 2241, 15361, 1276, 8, 13880, 6084, 43105, 6, 6230, 11354, 8317, 2721, 1429, 8093, 49, 1739, 5, 14, 14144, 38, 898, 6417, 15, 1152, 5498, 6, 1575, 62742, 5, 178, 40359, 24395, 13, 12641, 6330, 71, 248, 83, 804, 1062, 2297, 4, 1]
Second sample's labels (after -100 masking):
 [1566, 5606, 30728, 16, 2729, 3064, 708, 3, 7801, 14184, 8, 522, 3, 38, 99685, 551, 17053, 16, 6492, 9096, 487, 15, 5331, 5, 414, 70, 2168, 842, 14421, 2776, 898, 5, 10635, 351, 47247, 25782, 5, 27980, 38, 898, 3966, 262, 29172, 3156, 46858, 106997, 28, 66537, 6408, 4, 1]

Decoded summary for first sample's labels:
 بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل من جمهورية الجبل الأسود حاول نقل أسلحة إلى فرنسا، وقبضت الشرطة الألمانية على الرجل بالقرب من الحدود النمساوية، حيث أعترف بعلمه بنقل الأسلحة إلا أنه لم يكن يعرف السبب.
Decoded summary for second sample's labels:
 أعلن حاكم كارولاينا الشمالية حالة الطورائ في مدينة تشارلوت إضطرابات لليوم الثاني على التوالي، وذلك

In [24]:
split_datasets = tokenized_ds['train'].train_test_split(test_size=0.1, seed=42)
train_dataset = split_datasets['train']
eval_dataset = split_datasets['test']

print(f"Training dataset size: {len(train_dataset)}")
print(f"Evaluation dataset size: {len(eval_dataset)}")

Training dataset size: 44642
Evaluation dataset size: 4961


In [25]:
import shutil
corrupted = f"{SAVE_DIR}/arata5_results_v2/checkpoint-2500"
if os.path.exists(corrupted):
    shutil.rmtree(corrupted)
    print("Deleted corrupted checkpoint-2500")

In [26]:
corrupted = f"{SAVE_DIR}/arata5_results_v2/checkpoint-2500"
if os.path.exists(corrupted):
    shutil.rmtree(corrupted)
    print("Deleted corrupted checkpoint-2500")

In [27]:
import os

CHECKPOINT_DIR = f"{SAVE_DIR}/arata5_results_v2"

def check_checkpoint(path):
    """Check checkpoint status without deleting anything."""
    safetensors = os.path.join(path, "model.safetensors")
    pytorch = os.path.join(path, "pytorch_model.bin")

    if os.path.exists(safetensors):
        size = os.path.getsize(safetensors)
        return f"model.safetensors: {size/1e6:.1f} MB"
    elif os.path.exists(pytorch):
        size = os.path.getsize(pytorch)
        return f"pytorch_model.bin: {size/1e6:.1f} MB"
    else:
        return "No model weights found"

if os.path.isdir(CHECKPOINT_DIR):
    for item in sorted(os.listdir(CHECKPOINT_DIR)):
        item_path = os.path.join(CHECKPOINT_DIR, item)
        if not item.startswith('checkpoint-'):
            continue
        status = check_checkpoint(item_path)
        print(f"📁 {item} — {status}")
else:
    print("Directory not found")

📁 checkpoint-3500 — model.safetensors: 2145.6 MB
📁 checkpoint-4000 — model.safetensors: 2145.6 MB
📁 checkpoint-4500 — model.safetensors: 2145.6 MB


In [15]:
import os

CHECKPOINT_DIR = f"{SAVE_DIR}/arata5_results_v2"

if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')]
    checkpoints.sort(key=lambda x: int(x.split('-')[-1]))

    print("Current checkpoints on Drive:")
    for cp in checkpoints:
        cp_path = os.path.join(CHECKPOINT_DIR, cp)
        size = sum(os.path.getsize(os.path.join(dirpath, f))
                  for dirpath, _, files in os.walk(cp_path)
                  for f in files)
        status = "✅" if size > 500_000_000 else "❌"
        print(f"  {status} {cp} — {size/1e6:.1f} MB")

    if checkpoints:
        latest = checkpoints[-1]
        latest_step = int(latest.split('-')[-1])
        print(f"\nLatest saved step: {latest_step}")
        print(f"Training is at step: ~3500")

        if latest_step >= 3500:
            print("✅ Checkpoint-3500 (or newer) exists!")
        else:
            print("⚠️ Latest checkpoint is behind training. Wait for the next save.")
else:
    print("❌ Directory not found!")

Current checkpoints on Drive:
  ✅ checkpoint-3500 — 5768.2 MB
  ✅ checkpoint-4000 — 5768.2 MB
  ✅ checkpoint-4500 — 5768.2 MB

Latest saved step: 4500
Training is at step: ~3500
✅ Checkpoint-3500 (or newer) exists!


In [14]:
from transformers import TrainingArguments, Trainer, DataCollatorForSeq2Seq
import os

CHECKPOINT_DIR = f"{SAVE_DIR}/arata5_results_v2"

last_checkpoint = None
if os.path.isdir(CHECKPOINT_DIR):
    checkpoints = [d for d in os.listdir(CHECKPOINT_DIR) if d.startswith('checkpoint-')]
    if checkpoints:
        checkpoints.sort(key=lambda x: int(x.split('-')[-1]))
        last_checkpoint = os.path.join(CHECKPOINT_DIR, checkpoints[-1])
        print(f"Found checkpoint: {last_checkpoint}")
        print("Resuming training...")
    else:
        print("No checkpoint found. Starting from scratch.")
else:
    print("No checkpoint directory. Starting from scratch.")

training_args = TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    eval_strategy="steps",
    eval_steps=500,
    learning_rate=3e-5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    logging_steps=50,
    save_steps=500,
    save_total_limit=3,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_grad_norm=1.0,
    warmup_steps=500,
    weight_decay=0.01,
    report_to="none"
)

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

if last_checkpoint:
    trainer.train(resume_from_checkpoint=last_checkpoint)
else:
    trainer.train()

trainer.save_model(f"{CHECKPOINT_DIR}/final_model")
print("Saved final model")

No checkpoint found. Starting from scratch.


Step,Training Loss,Validation Loss
500,35.649875,8.420784
1000,30.134253,6.938391


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Step,Training Loss,Validation Loss
500,35.649875,8.420784
1000,30.134253,6.938391
1500,27.474749,6.152627
2000,25.840981,5.695333
2500,24.443486,5.362844
3000,23.511533,5.111002
3500,22.727036,4.925842
4000,22.188079,4.759251
4500,21.557290,4.640172


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [28]:
FINAL_DIR = f"{SAVE_DIR}/final_model_for_delivery"
trainer.save_model(FINAL_DIR)
print(f"✅ Final model saved to: {FINAL_DIR}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Final model saved to: /content/drive/MyDrive/arabic_summarization/final_model_for_delivery


In [36]:
from transformers import AutoModelForSeq2SeqLM

model_path = f"{SAVE_DIR}/final_model_for_delivery"
model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model.config.tie_word_embeddings = False

FINAL_DIR = f"{SAVE_DIR}/final_model_for_delivery_clean"
model.save_pretrained(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print(f"✅ Saved clean model to: {FINAL_DIR}")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Saved clean model to: /content/drive/MyDrive/arabic_summarization/final_model_for_delivery_clean


In [29]:
import torch
import gc

gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

print(f"Allocated: {torch.cuda.memory_allocated()/1e9:.2f} GB")
print(f"Reserved:  {torch.cuda.memory_reserved()/1e9:.2f} GB")

Allocated: 9.78 GB
Reserved:  10.39 GB


In [38]:
!pip install -q rouge-score

from rouge_score import rouge_scorer
import torch

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)

def evaluate_rouge(model, tokenizer, dataset, num_samples=100, device=device):
    model.eval()
    predictions, references = [], []

    for i in range(min(num_samples, len(dataset))):
        sample = dataset[i]
        input_ids = torch.tensor([sample['input_ids']]).to(device)

        with torch.no_grad():
            outputs = model.generate(
                input_ids,
                max_length=128,
                num_beams=4,
                early_stopping=True
            )

        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)
        ref = tokenizer.decode([t for t in sample['labels'] if t != -100], skip_special_tokens=True)

        predictions.append(pred)
        references.append(ref)

    scores = {'rouge1': [], 'rouge2': [], 'rougeL': []}
    for pred, ref in zip(predictions, references):
        score = scorer.score(ref, pred)
        for key in scores:
            scores[key].append(score[key].fmeasure)

    return {k: np.mean(v) for k, v in scores.items()}, predictions, references

# Evaluate extractive baseline first
print("Evaluating extractive baseline...")
extractive_scores = []
for i in range(min(100, len(eval_dataset))):
    # Get original article from eval dataset
    sample = ds['train'][eval_dataset[i]['__index_level_0__']] if '__index_level_0__' in eval_dataset[i] else None
    if sample is None:
        continue
    pred = extractive_summary(sample['article'], top_k=3)
    ref = sample['summary']
    score = scorer.score(ref, pred)
    extractive_scores.append({
        'rouge1': score['rouge1'].fmeasure,
        'rouge2': score['rouge2'].fmeasure,
        'rougeL': score['rougeL'].fmeasure
    })

if extractive_scores:
    print("Extractive Baseline ROUGE:")
    print(f"  ROUGE-1: {np.mean([s['rouge1'] for s in extractive_scores]):.4f}")
    print(f"  ROUGE-2: {np.mean([s['rouge2'] for s in extractive_scores]):.4f}")
    print(f"  ROUGE-L: {np.mean([s['rougeL'] for s in extractive_scores]):.4f}")

Evaluating extractive baseline...


In [35]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
import json
import os

model_path = f"{SAVE_DIR}/final_model_for_delivery"

model = AutoModelForSeq2SeqLM.from_pretrained(model_path)
model.config.tie_word_embeddings = False

model.config.save_pretrained(model_path)

config_path = os.path.join(model_path, "config.json")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

config["tie_word_embeddings"] = False

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("✅ Config fixed. Warnings will be gone for your client.")

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Config fixed. Warnings will be gone for your client.


In [34]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from datasets import load_dataset
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_path = f"{SAVE_DIR}/final_model_for_delivery"

# Load model
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(device)
model.config.tie_word_embeddings = False

def summarize_arabic(text, max_length=128):
    inputs = tokenizer("summarize: " + text, return_tensors="pt",
                      max_length=512, truncation=True).to(device)

    outputs = model.generate(
        **inputs,
        max_length=max_length,
        min_length=20,
        num_beams=6,
        length_penalty=1.2,
        early_stopping=True,
        no_repeat_ngram_size=3,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

ds = load_dataset("arbml/AraSum")

test_example = ds['train'][0]
test_article = test_example['article']
reference_summary = test_example['summary']

print("=" * 60)
print("ORIGINAL ARTICLE:")
print("=" * 60)
print(test_article[:500] + "..." if len(test_article) > 500 else test_article)

print("\n" + "=" * 60)
print("REFERENCE SUMMARY (Human-written):")
print("=" * 60)
print(reference_summary)

print("\n" + "=" * 60)
print("GENERATED SUMMARY (Your Model):")
print("=" * 60)
print(summarize_arabic(test_article))

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


ORIGINAL ARTICLE:
"بدأت اليوم الجمعة( 23 أيلول/ سبتمبر 2016 ) في ميونيخ محاكمة رجل من جمهورية الجبل الأسود ( مونتينيغرو)، اعتقل في ألمانيا للاشتباه في انه كان ينقل أسلحة قبل أيام من اعتداءات باريس التي حصلت في تشرين الثاني/نوفمبر العام الماضي. وتريد المحكمة معرفة عما إذا كانت هذه الأسلحة أعدت للاستخدام في هجمات فرنسا. وقالت المتحدثة إن المتهم ""اعترف أنه كان على علم بوجود أسلحة في سيارته، لكنه لا يعرف ما إذا كانت ستستخدم لتنفيذ اعتداء"". وأضافت المتحدثة أن الرجل الذي يبلغ الحادية والخمسين ويدعى فوسيليتش، ملاحق ""...

REFERENCE SUMMARY (Human-written):
بدأت محكمة ميونخ النظر في اتهامات متعلقة برجل من جمهورية الجبل الأسود حاول نقل أسلحة إلى فرنسا، وقبضت الشرطة الألمانية على الرجل بالقرب من الحدود النمساوية، حيث أعترف بعلمه بنقل الأسلحة إلا أنه لم يكن يعرف السبب.

GENERATED SUMMARY (Your Model):
"كشفت الشرطة الألمانية أن رجل من دول دول القرن الأسود احتجزتها الشرطة في ألمانيا بتهمة ""داعش""، فيما أعلنت الشرطة أن المشتبه في أن لـ""إرهابية"" في باريس، فيما أعلن المتهم أن هذه الأسلحة تستخدم ف